<div style="font-size: 13px; line-height: 1.25;">

# DTI Regulation Mapping × Comtrade Services Trade — Merge Script

This notebook loads the **DTI (Digital Trade Integration) regulation-to-BPM6 mapping file**, parses dates, builds regulation activity timelines, and merges the resulting flags into the **UN Comtrade bilateral services trade dataset**.

The final dataset has **three column blocks**:
- **Trade data** — original Comtrade bilateral services import data
- **Negative effect flags + counts** — restrictive regulations mapped per pillar/subpillar
- **Positive effect flags + counts** — enabling regulations mapped per pillar/subpillar

</div>

<div style="font-size: 25px; line-height: 1.25;">
SECTION 0 — SETUP
</div>

In [1]:
required_packages <- c("readxl", "dplyr", "tidyr", "lubridate", "stringr", "purrr")
for (pkg in required_packages) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    install.packages(pkg, repos = "https://cloud.r-project.org")
  }
  library(pkg, character.only = TRUE)
}

options(jupyter.rich_display = FALSE)
cat("All packages loaded.\n")

# --- Configuration block ------------------------------------------------------

# DTI mapping file path (disaggregated telecom version)
DTI_FILE <- "../data/mapping/dti_bpm6_mapping_final.xlsx"
DTI_SHEET <- "Regulation Mapping"
DTI_REASONING_SHEET <- "Reasoning"
EU_DISAGGREGATION_SHEET <- "EU_Disaggregation_Reasoning"

# Comtrade data: if full_df exists in environment, use it; otherwise load CSV
COMTRADE_CSV <- "../data/raw/data_result.csv"

# Output path for final merged dataset
OUTPUT_MERGED <- "../data/final_dataset/comtrade_dti_final.csv"

Warning message:
"package 'readxl' was built under R version 4.5.1"

Attaching package: 'dplyr'


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union


Warning message:
"package 'lubridate' was built under R version 4.5.1"

Attaching package: 'lubridate'


The following objects are masked from 'package:base':

    date, intersect, setdiff, union




All packages loaded.


<div style="font-size: 25px; line-height: 1.25;">
SECTION 1 — LOAD AND INSPECT DTI MAPPING FILE
</div>

In [2]:
cat("\n--- Loading DTI Regulation Mapping ---\n")
dti <- readxl::read_excel(DTI_FILE, sheet = DTI_SHEET)

# ▶ MODIFICATION 1 — attach the original Excel row number for each row, so we
# can match against EU_Disaggregation_Reasoning$Excel_Row_In_Main_Sheet later
# regardless of subsequent filtering, sorting, or duplication.
# Excel data starts at row 2 (header at row 1).
dti$dti_excel_row <- seq_len(nrow(dti)) + 1L

# Fix HTML entities in text columns (Excel has &amp; instead of &)
dti$Pillar    <- gsub("&amp;", "&", dti$Pillar, fixed = TRUE)
dti$Subpillar <- gsub("&amp;", "&", dti$Subpillar, fixed = TRUE)

cat(sprintf("Dimensions: %d rows x %d columns\n", nrow(dti), ncol(dti)))
cat("\nColumn names:\n")
cat(paste(" ", names(dti), collapse = "\n"), "\n")
cat("\nSample rows (first 3):\n")
print(head(dti, 3))

# Load Reasoning sheet for reference (not used in merge)
cat("\n--- Loading Reasoning sheet (for reference only) ---\n")
dti_reasoning <- readxl::read_excel(DTI_FILE, sheet = DTI_REASONING_SHEET)
cat(sprintf("Reasoning sheet: %d rows x %d columns\n",
            nrow(dti_reasoning), ncol(dti_reasoning)))

# Quick summary
cat(sprintf("\nUnique countries: %d\n", n_distinct(dti$Country)))
cat(sprintf("Unique pillars:  %d\n", n_distinct(dti$Pillar)))
cat(sprintf("Unique subpillars: %d\n", n_distinct(dti$Subpillar)))

# Identify the BPM6 service/goods categories from _effect columns
effect_cols <- grep("_effect$", names(dti), value = TRUE)
bpm6_categories <- sub("_effect$", "", effect_cols)
cat(sprintf("\nBPM6 categories found: %d\n", length(bpm6_categories)))
cat(paste("  ", bpm6_categories, collapse = "\n"), "\n")

# Note Partner_Specific column
cat(sprintf("\nPartner_Specific flag: %d regulations are partner-specific\n",
            sum(dti$Partner_Specific == 1, na.rm = TRUE)))

# ▶ MODIFICATION 3 — sanity-check that the new Disaggregate column exists
if ("Disaggregate" %in% names(dti)) {
  cat(sprintf("Disaggregate flag: %d regulations flagged (1) for disaggregation\n",
              sum(dti$Disaggregate == 1, na.rm = TRUE)))
}


--- Loading DTI Regulation Mapping ---
Dimensions: 6992 rows x 52 columns

Column names:
  Country
  Pillar
  Subpillar
  Subpillar_ID
  Act_Practice
  Coverage
  Description
  Timeframe
  Enabling_or_Restriction
  Partner_Specific
  Date_Entry_Into_Force
  Date_Enacted_First_Reported
  Date_Last_Amended
  Expiry_End_Date
  Date_Last_Extension
  Other_Date
  General_merchandise_effect
  General_merchandise_direction
  Goods_under_merchanting_effect
  Goods_under_merchanting_direction
  Nonmonetary_gold_effect
  Nonmonetary_gold_direction
  Manufacturing_services_effect
  Manufacturing_services_direction
  Maintenance_repair_services_effect
  Maintenance_repair_services_direction
  Transport_effect
  Transport_direction
  Travel_effect
  Travel_direction
  Construction_effect
  Construction_direction
  Insurance_pension_services_effect
  Insurance_pension_services_direction
  Financial_services_effect
  Financial_services_direction
  IP_charges_effect
  IP_charges_direction
  Telecom_9

<div style="font-size: 25px; line-height: 1.25;">
SECTION 2 — DATE COLUMN PARSING AND STANDARDISATION
</div>

In [3]:
cat("\n--- Parsing date columns ---\n")

# Define which date columns to parse (exclude Other_Date)
date_cols_to_parse <- c("Date_Entry_Into_Force", "Date_Enacted_First_Reported",
                        "Date_Last_Amended", "Expiry_End_Date", "Date_Last_Extension")

# Parsing function: handles yyyy, mm/yyyy, dd/mm/yyyy, and numeric year
parse_flexible_date <- function(x) {
  if (is.na(x) || x == "" || x == "NA") {
    return(list(date = as.Date(NA), precision = NA_character_))
  }

  x_str <- as.character(x)

  # Handle pure numeric (e.g. 2015 or 2015.0 from Excel)
  x_clean <- sub("\\.0$", "", x_str)

  if (grepl("^\\d{4}$", x_clean)) {
    return(list(date = as.Date(paste0(x_clean, "-01-01")), precision = "year"))
  } else if (grepl("^\\d{1,2}/\\d{4}$", x_clean)) {
    parts <- strsplit(x_clean, "/")[[1]]
    month <- sprintf("%02d", as.integer(parts[1]))
    year  <- parts[2]
    return(list(date = as.Date(paste0(year, "-", month, "-01")), precision = "month"))
  } else if (grepl("^\\d{1,2}/\\d{1,2}/\\d{4}$", x_clean)) {
    parts <- strsplit(x_clean, "/")[[1]]
    day   <- sprintf("%02d", as.integer(parts[1]))
    month <- sprintf("%02d", as.integer(parts[2]))
    year  <- parts[3]
    return(list(date = as.Date(paste0(year, "-", month, "-", day)), precision = "full"))
  } else {
    return(list(date = as.Date(NA), precision = NA_character_))
  }
}

# Apply parsing to each date column
for (col in date_cols_to_parse) {
  parsed <- lapply(dti[[col]], parse_flexible_date)
  dti[[paste0(col, "_parsed")]]    <- as.Date(sapply(parsed, function(p) p$date),
                                               origin = "1970-01-01")
  dti[[paste0(col, "_precision")]] <- sapply(parsed, function(p) p$precision)
}

# Verification: check for parsing failures (had a value but parsed to NA)
cat("\nParsing verification — values that had content but failed to parse:\n")
for (col in date_cols_to_parse) {
  original_non_na <- sum(!is.na(dti[[col]]) & dti[[col]] != "")
  parsed_non_na   <- sum(!is.na(dti[[paste0(col, "_parsed")]]))
  failures        <- original_non_na - parsed_non_na
  cat(sprintf("  %-30s: %d non-blank → %d parsed (%d failures)\n",
              col, original_non_na, parsed_non_na, failures))
}


--- Parsing date columns ---

Parsing verification — values that had content but failed to parse:
  Date_Entry_Into_Force         : 4157 non-blank → 4157 parsed (0 failures)
  Date_Enacted_First_Reported   : 1079 non-blank → 1079 parsed (0 failures)
  Date_Last_Amended             : 1182 non-blank → 1182 parsed (0 failures)
  Expiry_End_Date               : 1231 non-blank → 1231 parsed (0 failures)
  Date_Last_Extension           : 15 non-blank → 15 parsed (0 failures)


<div style="font-size: 25px; line-height: 1.25;">
SECTION 3 — HARDCODE PILLAR AND SUBPILLAR REFERENCE TABLE
</div>

In [4]:
# --- 3.1 Subpillar-to-column mapping for Pillar 6 ----------------------------

pillar6_subpillar_map <- data.frame(
  subpillar_text = c(
    "Ban to transfer and local processing requirement",
    "Local storage requirement",
    "Infrastructure requirement",
    "Conditional flow regime",
    "Participation in trade agreements committing to open cross-border data flows"
  ),
  col_id = c("p6_1", "p6_2", "p6_3", "p6_4", "p6_5"),
  col_label = c(
    "6.1 Ban on transfer and local processing requirement",
    "6.2 Local storage requirement",
    "6.3 Infrastructure requirement",
    "6.4 Conditional flow regime",
    "6.5 Participation in trade agreements committing to open cross-border data flows"
  ),
  stringsAsFactors = FALSE
)

# --- 3.2 Pillar-to-column mapping for all other pillars -----------------------

pillar_map <- data.frame(
  pillar_text = c(
    "Tariffs and trade defence measures applied on ICT goods",
    "Public procurement of ICT goods and online services",
    "Foreign Direct Investment (FDI) in sectors relevant to digital trade",
    "Intellectual Property Rights (IPRs)",
    "Telecom infrastructure & competition",
    "Domestic data policies",
    "Intermediary liability",
    "Content access",
    "Quantitative trade restrictions for ICT goods and online services",
    "Technical standards applied to ICT goods and online services",
    "Online sales and transactions"
  ),
  col_id = c("p1", "p2", "p3", "p4", "p5", "p7", "p8", "p9", "p10", "p11", "p12"),
  col_label = c(
    "1. Tariffs and trade defence measures applied to ICT goods",
    "2. Public procurement of ICT goods and online services",
    "3. Foreign direct investment in sectors relevant to digital trade",
    "4. Intellectual Property Rights (IPRs)",
    "5. Telecom infrastructure and competition",
    "7. Domestic data policies",
    "8. Intermediary liability",
    "9. Content access",
    "10. Quantitative trade restrictions for ICT goods and online services",
    "11. Technical standards applied to ICT goods and online services",
    "12. Online sales and transactions"
  ),
  stringsAsFactors = FALSE
)

# --- 3.3 Combined reference table of all 16 output columns --------------------

all_flag_cols <- c(pillar6_subpillar_map$col_id, pillar_map$col_id)

cat("\n--- 16 Output Flag Columns ---\n")
cat("Cross-border data policy subpillars (individual):\n")
for (i in seq_len(nrow(pillar6_subpillar_map))) {
  cat(sprintf("  %s — %s\n", pillar6_subpillar_map$col_id[i],
              pillar6_subpillar_map$col_label[i]))
}
cat("Other pillars (aggregated):\n")
for (i in seq_len(nrow(pillar_map))) {
  cat(sprintf("  %s — %s\n", pillar_map$col_id[i], pillar_map$col_label[i]))
}

# --- 3.4 Assign each DTI regulation to its output column ----------------------

dti$col_assignment <- NA_character_

for (i in seq_len(nrow(pillar6_subpillar_map))) {
  mask <- dti$Pillar == "Cross-border data policies" &
          dti$Subpillar == pillar6_subpillar_map$subpillar_text[i]
  dti$col_assignment[mask] <- pillar6_subpillar_map$col_id[i]
}

for (i in seq_len(nrow(pillar_map))) {
  mask <- dti$Pillar == pillar_map$pillar_text[i] & is.na(dti$col_assignment)
  dti$col_assignment[mask] <- pillar_map$col_id[i]
}

unassigned <- sum(is.na(dti$col_assignment))
if (unassigned > 0) {
  cat(sprintf("\n⚠ %d regulations could not be assigned to a column.\n", unassigned))
  cat("  Unassigned pillars:\n")
  print(table(dti$Pillar[is.na(dti$col_assignment)]))
} else {
  cat(sprintf("\nAll %d regulations assigned to output columns.\n", nrow(dti)))
}

cat("\nRegulations per output column:\n")
print(table(dti$col_assignment))


--- 16 Output Flag Columns ---
Cross-border data policy subpillars (individual):
  p6_1 — 6.1 Ban on transfer and local processing requirement
  p6_2 — 6.2 Local storage requirement
  p6_3 — 6.3 Infrastructure requirement
  p6_4 — 6.4 Conditional flow regime
  p6_5 — 6.5 Participation in trade agreements committing to open cross-border data flows
Other pillars (aggregated):
  p1 — 1. Tariffs and trade defence measures applied to ICT goods
  p2 — 2. Public procurement of ICT goods and online services
  p3 — 3. Foreign direct investment in sectors relevant to digital trade
  p4 — 4. Intellectual Property Rights (IPRs)
  p5 — 5. Telecom infrastructure and competition
  p7 — 7. Domestic data policies
  p8 — 8. Intermediary liability
  p9 — 9. Content access
  p10 — 10. Quantitative trade restrictions for ICT goods and online services
  p11 — 11. Technical standards applied to ICT goods and online services
  p12 — 12. Online sales and transactions

All 6992 regulations assigned to output c

<div style="font-size: 25px; line-height: 1.25;">
SECTION 4 — BUILD REGULATION ACTIVITY TIMELINE PER COUNTRY
</div>

In [5]:
cat("\n--- Building regulation activity timelines ---\n")

# Helper: extract activation year from a parsed date + precision
get_activation_year <- function(date_val, precision) {
  if (is.na(date_val)) return(NA_integer_)
  yr <- as.integer(format(date_val, "%Y"))
  mo <- as.integer(format(date_val, "%m"))
  if (!is.na(precision) && precision %in% c("month", "full") && mo >= 7) {
    return(yr + 1L)
  }
  return(yr)
}

# Helper: extract expiry year
get_expiry_year <- function(date_val, precision) {
  if (is.na(date_val)) return(NA_integer_)
  yr <- as.integer(format(date_val, "%Y"))
  mo <- as.integer(format(date_val, "%m"))
  if (!is.na(precision) && precision %in% c("month", "full") && mo <= 6) {
    return(yr - 1L)
  }
  return(yr)
}

# Compute activation year
dti$activation_year <- mapply(function(d, p) get_activation_year(d, p),
                               dti$Date_Entry_Into_Force_parsed,
                               dti$Date_Entry_Into_Force_precision)

# Fall back to Date_Enacted if Entry_Into_Force is missing
fallback_mask <- is.na(dti$activation_year) & !is.na(dti$Date_Enacted_First_Reported_parsed)
dti$activation_year[fallback_mask] <- mapply(
  function(d, p) get_activation_year(d, p),
  dti$Date_Enacted_First_Reported_parsed[fallback_mask],
  dti$Date_Enacted_First_Reported_precision[fallback_mask]
)

# Compute expiry year
dti$expiry_year <- mapply(function(d, p) get_expiry_year(d, p),
                           dti$Expiry_End_Date_parsed,
                           dti$Expiry_End_Date_precision)

# --- Classify regulations as negative, positive, or both ---------------------

negative_labels <- c("Regulatory restriction",
                     "Tariffs - Regulatory restriction",
                     "Tariff header and regulatory restriction")

positive_labels <- c("Enabling measure",
                     "Tariffs - Enabling measure",
                     "Tariff header and enabling measure")

both_labels <- c("Tariffs - Regulatory restriction and enabling measure")

dti$direction_class <- case_when(
  dti$Enabling_or_Restriction %in% negative_labels ~ "negative",
  dti$Enabling_or_Restriction %in% positive_labels ~ "positive",
  dti$Enabling_or_Restriction %in% both_labels     ~ "both",
  TRUE ~ NA_character_
)

cat("\nDirection classification:\n")
print(table(dti$direction_class, useNA = "ifany"))

# For "both" regulations: duplicate them — one copy as negative, one as positive
both_regs <- dti %>% filter(direction_class == "both")
if (nrow(both_regs) > 0) {
  both_neg <- both_regs %>% mutate(direction_class = "negative")
  both_pos <- both_regs %>% mutate(direction_class = "positive")
  dti <- dti %>%
    filter(direction_class != "both") %>%
    bind_rows(both_neg, both_pos)
  cat(sprintf("  'Both' regulations duplicated: %d → %d neg + %d pos\n",
              nrow(both_regs), nrow(both_neg), nrow(both_pos)))
}

excluded_count <- sum(is.na(dti$activation_year) & !is.na(dti$col_assignment))
cat(sprintf("\nRegulations with activation year: %d\n",
            sum(!is.na(dti$activation_year))))
cat(sprintf("Regulations excluded (no date):    %d\n", excluded_count))
cat(sprintf("Regulations with expiry date:      %d\n",
            sum(!is.na(dti$expiry_year))))

dti_active <- dti %>%
  filter(!is.na(col_assignment), !is.na(activation_year), !is.na(direction_class))

cat(sprintf("\nActive regulations for timeline (pre-filter): %d\n", nrow(dti_active)))
cat(sprintf("  Negative: %d\n", sum(dti_active$direction_class == "negative")))
cat(sprintf("  Positive: %d\n", sum(dti_active$direction_class == "positive")))

# ============================================================================
# Treatment-definition filtering (P6 only) -- Option 3 per supervisor sign-off
#
# Pillar 6 (Cross-border data policies) is the focus of the thesis. We refine
# the P6 treatment indicator in two ways:
#
# (a) Coverage whitelist. Restrict P6 regulations to those whose DTI Coverage
#     classification plausibly bites on a BPM6 services category in the panel.
#     This excludes sector-narrow regulations (Health, Public, Gambling, etc.)
#     whose impact on bilateral services trade would be near-zero, reducing
#     measurement error in the binary treatment indicator.
#
# (b) Manual exclusion of US Team Telecom (1999). DTI's own description of
#     this regulation states "The United States has not adopted laws or
#     regulations requiring that data be stored locally." Team Telecom is
#     informal interagency administrative practice rather than codified law,
#     so excluding it removes a known false-positive treatment event.
#
# Other pillars (P1-P5, P7-P12) are NOT filtered -- their Coverage scope
# is interpreted differently because the dependent variable isn't tied to
# their sector breadth in the same way.
# ============================================================================

p6_allowed_coverages <- c(
  "Horizontal",
  "Telecommunications sector",
  "Cloud-computing sector",
  "Electronic communications sector",
  "Financial sector",
  "Financial services",
  "Insurance sector"
)

# Identify P6 rows in dti_active. Use the col_assignment field (which contains
# "p6_1" through "p6_5" for Pillar-6 subpillars).
is_p6_row <- grepl("^p6_", dti_active$col_assignment)

# Identify the USA Team Telecom regulation. Match on Country + Subpillar_ID
# + the 1999 Date_Entry_Into_Force string. Use defensive equality so a
# missing match leaves the regulation in (we'd rather over-include than
# silently over-exclude on this rule).
is_us_team_telecom <- (
  dti_active$Country == "UNITED STATES" &
  !is.na(dti_active$Subpillar_ID) & dti_active$Subpillar_ID == 6.2 &
  !is.na(dti_active$Date_Entry_Into_Force) &
  grepl("10/1999", dti_active$Date_Entry_Into_Force)
)

# For each P6 row: keep it iff Coverage is in the whitelist AND it isn't
# Team Telecom. Non-P6 rows are kept regardless.
keep_p6_row <- is_p6_row &
               dti_active$Coverage %in% p6_allowed_coverages &
               !is_us_team_telecom

# Final keep mask: non-P6 rows OR (P6 rows that survive the filter)
keep_mask <- (!is_p6_row) | keep_p6_row

n_p6_before        <- sum(is_p6_row)
n_p6_after         <- sum(keep_p6_row)
n_dropped_coverage <- sum(is_p6_row & !(dti_active$Coverage %in% p6_allowed_coverages))
n_dropped_team_tel <- sum(is_p6_row & is_us_team_telecom &
                            dti_active$Coverage %in% p6_allowed_coverages)

dti_active <- dti_active[keep_mask, ]

cat(sprintf("\nP6 treatment-definition filtering (Option 3):\n"))
cat(sprintf("  P6 regulations before filter:    %d\n", n_p6_before))
cat(sprintf("    Dropped (Coverage not on whitelist):  %d\n", n_dropped_coverage))
cat(sprintf("    Dropped (US Team Telecom exclusion):  %d\n", n_dropped_team_tel))
cat(sprintf("  P6 regulations after filter:     %d\n", n_p6_after))
cat(sprintf("\nActive regulations for timeline (post-filter): %d\n", nrow(dti_active)))


--- Building regulation activity timelines ---

Direction classification:

    both negative positive 
      24     5290     1678 
  'Both' regulations duplicated: 24 → 24 neg + 24 pos

Regulations with activation year: 5236
Regulations excluded (no date):    1780
Regulations with expiry date:      1231

Active regulations for timeline (pre-filter): 5236
  Negative: 3712
  Positive: 1524

P6 treatment-definition filtering (Option 3):
  P6 regulations before filter:    344
    Dropped (Coverage not on whitelist):  118
    Dropped (US Team Telecom exclusion):  1
  P6 regulations after filter:     225

Active regulations for timeline (post-filter): 5117


<div style="font-size: 20px; line-height: 1.25;">
SECTION 4B — DISAGGREGATE EUROPEAN UNION REGULATIONS TO MEMBER STATES
</div>

<!-- ▶ MODIFICATION 1 — replaces the previous blanket EU expansion. -->

The previous version of this cell expanded every `EUROPEAN UNION` row onto all
27 member states unconditionally. That double-counts content for regulations
already captured at member level (e.g., GDPR appears in every member state's
own row under *Framework for data protection*; Trade Secrets Directive is
transposed at every member's row; the tariff-statistic rows have country-specific
WITS values for each member that differ from the EU aggregate).

This section now reads the `EU_Disaggregation_Reasoning` sheet, which contains
one row per EU regulation with two governing fields:

* **`Excel_Row_In_Main_Sheet`** — identifies each EU regulation by its row
  position in the `Regulation Mapping` sheet at load time
* **`Recommended_Action_If_Disaggregating`** — either a comma-separated list of
  country names to disaggregate to, or `"None — do not disaggregate to any
  member state."` for rows that should not be expanded

For each EU regulation: if the action is `None`, the EU row stays as-is in the
final dataset under `Country = "EUROPEAN UNION"` (and will get unmatched in the
Comtrade join because Comtrade has no `EUROPEAN UNION` reporter — that is the
correct behaviour, since these rows are already represented at the member-state
level via other rows in the dataset). If the action is a country list, the EU
row is replicated across exactly those countries; the list may be all 27 or any
subset (for example, the DSA partial-collision rows skip Germany and the NLF
row skips Poland).

Identification is done by Excel row number at the time the file was first loaded
(via the `dti_excel_row` index attached during Section 1), so the logic is
robust to any subsequent filtering or duplication of `dti_active`.

In [6]:
cat("\n--- Loading EU_Disaggregation_Reasoning sheet ---\n")
eu_disagg <- readxl::read_excel(DTI_FILE, sheet = EU_DISAGGREGATION_SHEET)
cat(sprintf("EU disaggregation sheet: %d rows x %d columns\n",
            nrow(eu_disagg), ncol(eu_disagg)))

# ============================================================================
# Manual override for GDPR disaggregation
# ----------------------------------------------------------------------------
# The EU_Disaggregation_Reasoning sheet's default directive for GDPR
# (Excel row 2826, EU-level Conditional flow regime, April 2016) is
# "None -- do not disaggregate", based on the assumption that each EU
# member state has its own GDPR-equivalent row in the main sheet.
#
# Verification of the workbook shows this assumption does not hold: the
# main sheet has GDPR-equivalent rows only for the UK (no longer EU),
# Norway (EEA), and Malta, not for the other 24 EU members. Leaving the
# directive as "None" results in GDPR contributing zero treatment to
# bilateral services-trade observations for EU member states across
# 2016-2024 -- substantively the wrong outcome.
#
# We override the directive to disaggregate GDPR to all 27 EU member
# states. The override is applied here in code rather than in the
# workbook so the source data is preserved and the change is auditable.
# ============================================================================

EU27_FOR_DISAGGREGATION <- c(
  "AUSTRIA", "BELGIUM", "BULGARIA", "CROATIA", "CYPRUS",
  "CZECH REPUBLIC", "DENMARK", "ESTONIA", "FINLAND", "FRANCE",
  "GERMANY", "GREECE", "HUNGARY", "IRELAND", "ITALY",
  "LATVIA", "LITHUANIA", "LUXEMBOURG", "MALTA", "NETHERLANDS",
  "POLAND", "PORTUGAL", "ROMANIA", "SLOVAKIA", "SLOVENIA",
  "SPAIN", "SWEDEN"
)
GDPR_EXCEL_ROW <- 2826L

gdpr_override_action <- paste(EU27_FOR_DISAGGREGATION, collapse = ", ")

gdpr_idx <- which(eu_disagg$Excel_Row_In_Main_Sheet == GDPR_EXCEL_ROW)
if (length(gdpr_idx) == 1L) {
  old_action <- eu_disagg$Recommended_Action_If_Disaggregating[gdpr_idx]
  eu_disagg$Recommended_Action_If_Disaggregating[gdpr_idx] <- gdpr_override_action
  cat(sprintf("\n[MANUAL OVERRIDE] GDPR (Excel row %d) directive changed:\n",
              GDPR_EXCEL_ROW))
  cat(sprintf("  Before: %s\n", substr(old_action, 1, 80)))
  cat(sprintf("  After:  disaggregate to all %d EU-27 member states\n",
              length(EU27_FOR_DISAGGREGATION)))
} else {
  warning(sprintf(
    "GDPR override skipped: Excel row %d not found in EU_Disaggregation_Reasoning (found %d matches)",
    GDPR_EXCEL_ROW, length(gdpr_idx)))
}

stopifnot("Excel_Row_In_Main_Sheet" %in% names(eu_disagg))
stopifnot("Recommended_Action_If_Disaggregating" %in% names(eu_disagg))
stopifnot("Disaggregate" %in% names(eu_disagg))

# Convenience: tally the directives by action type
none_pattern <- "^None"
n_none      <- sum(grepl(none_pattern, eu_disagg$Recommended_Action_If_Disaggregating))
n_to_disagg <- nrow(eu_disagg) - n_none
cat(sprintf("  EU regulations to disaggregate: %d\n", n_to_disagg))
cat(sprintf("  EU regulations to leave as-is:  %d\n", n_none))

# --- Parse Recommended_Action into a list of countries per directive ---------
parse_country_list <- function(action_text) {
  if (is.na(action_text) || grepl(none_pattern, action_text)) return(character(0))
  parts <- strsplit(action_text, ",", fixed = TRUE)[[1]]
  trimws(parts)
}

eu_disagg$countries <- lapply(eu_disagg$Recommended_Action_If_Disaggregating,
                               parse_country_list)

# --- Build directive map keyed by Excel row in main sheet --------------------
# Excel row N corresponds to dti row index N - 1 in R (header at Excel row 1,
# data starts at Excel row 2 = R index 1). We attached `dti_excel_row` to every
# row of `dti` in Section 1 so we can match safely even after filtering.

directive_map <- setNames(eu_disagg$countries, eu_disagg$Excel_Row_In_Main_Sheet)

# --- Apply disaggregation to dti_active --------------------------------------

# Snapshot baseline counts before changes
n_before <- nrow(dti_active)
n_eu_before <- sum(dti_active$Country == "EUROPEAN UNION")

# Split dti_active into: EU rows that need processing, and everything else
non_eu <- dti_active %>% filter(Country != "EUROPEAN UNION")
eu_rows_active <- dti_active %>% filter(Country == "EUROPEAN UNION")

cat(sprintf("\nEU-level rows in dti_active before disaggregation: %d\n",
            nrow(eu_rows_active)))

# Build replacement rows
keep_as_eu_list  <- list()
disaggregated_list <- list()
processed_directives <- 0
unmatched_eu_rows <- 0

for (i in seq_len(nrow(eu_rows_active))) {
  row_i <- eu_rows_active[i, ]
  excel_row_key <- as.character(row_i$dti_excel_row)

  countries <- directive_map[[excel_row_key]]

  if (is.null(countries)) {
    # Row is from EU but has no entry in the directive sheet — keep as-is and warn
    keep_as_eu_list[[length(keep_as_eu_list) + 1L]] <- row_i
    unmatched_eu_rows <- unmatched_eu_rows + 1L
    next
  }

  processed_directives <- processed_directives + 1L

  if (length(countries) == 0L) {
    # Action = "None" — keep the EU row exactly as it was (no expansion).
    keep_as_eu_list[[length(keep_as_eu_list) + 1L]] <- row_i
  } else {
    # Replicate the row across the listed countries
    expansion <- do.call(rbind, lapply(countries, function(ms) {
      tmp <- row_i
      tmp$Country <- ms
      tmp
    }))
    disaggregated_list[[length(disaggregated_list) + 1L]] <- expansion
  }
}

keep_as_eu        <- if (length(keep_as_eu_list) > 0)        bind_rows(keep_as_eu_list)        else eu_rows_active[0, ]
disaggregated_eu  <- if (length(disaggregated_list) > 0)     bind_rows(disaggregated_list)     else eu_rows_active[0, ]

dti_active <- bind_rows(non_eu, keep_as_eu, disaggregated_eu)

# --- Confirmation report ------------------------------------------------------

cat(sprintf("\nEU regulations processed via directive sheet: %d (of %d in sheet)\n",
            processed_directives, nrow(eu_disagg)))

if (unmatched_eu_rows > 0) {
  cat(sprintf("WARNING: %d EU rows in dti_active had no matching directive (kept as-is).\n",
              unmatched_eu_rows))
}

cat(sprintf("EU rows kept as-is (Country = EUROPEAN UNION): %d\n", nrow(keep_as_eu)))
cat(sprintf("Country-level rows created from disaggregation: %d\n", nrow(disaggregated_eu)))
cat(sprintf("\nTotal dti_active rows: %d (was %d before disaggregation)\n",
            nrow(dti_active), n_before))
cat(sprintf("  Negative direction: %d\n", sum(dti_active$direction_class == "negative")))
cat(sprintf("  Positive direction: %d\n", sum(dti_active$direction_class == "positive")))

if (nrow(disaggregated_eu) > 0) {
  cat("\n--- Sample of disaggregated rows (first 5) ---\n")
  sample_cols <- c("Country", "Pillar", "Subpillar", "direction_class",
                   "activation_year", "col_assignment")
  sample_cols <- intersect(sample_cols, names(disaggregated_eu))
  print(head(disaggregated_eu[, sample_cols], 5))
}


--- Loading EU_Disaggregation_Reasoning sheet ---
EU disaggregation sheet: 29 rows x 12 columns

[MANUAL OVERRIDE] GDPR (Excel row 2826) directive changed:
  Before: None — do not disaggregate to any member state.
  After:  disaggregate to all 27 EU-27 member states
  EU regulations to disaggregate: 23
  EU regulations to leave as-is:  6

EU-level rows in dti_active before disaggregation: 25

EU regulations processed via directive sheet: 25 (of 29 in sheet)
EU rows kept as-is (Country = EUROPEAN UNION): 5
Country-level rows created from disaggregation: 537

Total dti_active rows: 5634 (was 5117 before disaggregation)
  Negative direction: 4032
  Positive direction: 1602

--- Sample of disaggregated rows (first 5) ---
# A tibble: 5 × 6
  Country  Pillar       Subpillar direction_class activation_year col_assignment
  <chr>    <chr>        <chr>     <chr>                     <int> <chr>         
1 AUSTRIA  Online sale… Local pr… negative                   2023 p12           
2 BELGIUM  

<div style="font-size: 20px; line-height: 1.25;">
SECTION 4C — COUNTRY NAME MATCHING (DTI ↔ COMTRADE)
</div>

In [7]:
cat("\n--- Building country name matching table ---\n")

country_name_overrides <- data.frame(
  dti_name = c(
    "KOREA", "HONG KONG", "BRUNEI", "BOLIVIA", "VENEZUELA", "SYRIA",
    "VIETNAM", "TANZANIA", "RUSSIA", "CZECH REPUBLIC", "COTE D'IVOIRE",
    "DEMOCRATIC REPUBLIC OF CONGO (DRC)", "CONGO", "KYRGYZ REPUBLIC",
    "LAOS", "PALESTINE", "TAIWAN", "ESWATINI", "IRAN", "CABO VERDE",
    "SAO TOME AND PRINCIPE", "TIMOR-LESTE", "TÜRKIYE", "BAHAMAS",
    "GAMBIA", "COMOROS", "PHILIPPINES", "SURINAME", "SEYCHELLES", "SAINT LUCIA",
    "UNITED STATES"
  ),
  comtrade_name_upper = c(
    "REP. OF KOREA", "CHINA, HONG KONG SAR", "BRUNEI DARUSSALAM",
    "BOLIVIA (PLURINATIONAL STATE OF)", "VENEZUELA (BOLIVARIAN REPUBLIC OF)",
    "SYRIAN ARAB REPUBLIC", "VIET NAM", "UNITED REP. OF TANZANIA",
    "RUSSIAN FEDERATION", "CZECHIA", "CÔTE D'IVOIRE",
    "DEM. REP. OF THE CONGO", "CONGO", "KYRGYZSTAN",
    "LAO PEOPLE'S DEM. REP.", "STATE OF PALESTINE", "TAIPEI, CHINESE",
    "ESWATINI", "IRAN", "CABO VERDE",
    "SAO TOME AND PRINCIPE", "TIMOR-LESTE", "TÜRKIYE", "BAHAMAS",
    "GAMBIA", "COMOROS", "PHILIPPINES", "SURINAME", "SEYCHELLES", "SAINT LUCIA",
    "USA"
  ),
  stringsAsFactors = FALSE
)

dti_active$country_norm <- dti_active$Country
for (i in seq_len(nrow(country_name_overrides))) {
  mask <- dti_active$country_norm == country_name_overrides$dti_name[i]
  dti_active$country_norm[mask] <- country_name_overrides$comtrade_name_upper[i]
}

cat(sprintf("Country name overrides applied: %d rules\n",
            nrow(country_name_overrides)))


--- Building country name matching table ---
Country name overrides applied: 31 rules


<div style="font-size: 20px; line-height: 1.25;">
SECTION 4D — EBOPS-TO-BPM6 SERVICE CATEGORY MAPPING (DISAGGREGATED TELECOM)
</div>

In [8]:
cat("\n--- Building EBOPS → BPM6 mapping (disaggregated telecom) ---\n")

# EBOPS 2010 top-level codes → BPM6 (code 9 excluded — handled separately)
ebops2010_map <- data.frame(
  top_code = as.character(c(1:8, 10:12)),
  bpm6_category = c(
    "Manufacturing_services", "Maintenance_repair_services", "Transport",
    "Travel", "Construction", "Insurance_pension_services",
    "Financial_services", "IP_charges",
    "Other_business_services", "Personal_cultural_recreational_services",
    "Government_goods_services"
  ),
  stringsAsFactors = FALSE
)

# EBOPS 2010 S-prefix → BPM6 (SI excluded — handled separately)
ebops2010_s_map <- data.frame(
  s_prefix = c("SA", "SB", "SC", "SD", "SE", "SF", "SG", "SH", "SJ", "SK", "SL"),
  bpm6_category = c(
    "Manufacturing_services", "Maintenance_repair_services", "Transport",
    "Travel", "Construction", "Insurance_pension_services",
    "Financial_services", "IP_charges",
    "Other_business_services", "Personal_cultural_recreational_services",
    "Government_goods_services"
  ),
  stringsAsFactors = FALSE
)

# EBOPS 2002 range-based mapping (telecom disaggregated)
ebops2002_ranges <- data.frame(
  range_start = c(200, 236, 245, 249, 251, 253, 260, 262, 266, 272, 279, 287, 291),
  range_end   = c(235, 244, 248, 250, 252, 259, 261, 265, 271, 278, 286, 290, 299),
  bpm6_category = c(
    "Transport", "Travel",
    "Telecom_9_1",                      # 245-248: Telecommunications
    "Telecom_9_2",                      # 249-250: Computer services
    "Telecom_9_3",                      # 251-252: Information services
    "Construction",
    "Insurance_pension_services",
    "Financial_services",
    "Telecom_9_2",                      # 266-271: Computer & info (mostly computer)
    "IP_charges",
    "Other_business_services",
    "Personal_cultural_recreational_services",
    "Government_goods_services"
  ),
  stringsAsFactors = FALSE
)

# Master function: given a cmdCode and classificationCode, return bpm6_category
map_ebops_to_bpm6 <- function(cmd_code, classification) {
  if (is.na(cmd_code) || is.na(classification)) return(NA_character_)

  cmd_str <- as.character(cmd_code)

  # Skip known aggregates
  if (cmd_str %in% c("TOTAL", "S", "TOEB02")) return(NA_character_)

  # --- EBOPS 2010 (EB10, EB10S, EB) ---
  if (grepl("EB10|^EB$", classification)) {

    # Handle S-prefix codes (SA, SA1, SB, SC3A, SI1, SI21, etc.)
    if (grepl("^S[A-L]", cmd_str)) {
      if (grepl("^SI$", cmd_str)) return(NA_character_)  # SI aggregate → skip
      if (grepl("^SI1", cmd_str)) return("Telecom_9_1")  # SI1, SI11, SI111, etc.
      if (grepl("^SI2", cmd_str)) return("Telecom_9_2")  # SI2, SI21, etc.
      if (grepl("^SI3", cmd_str)) return("Telecom_9_3")  # SI3, SI31, etc.

      # All other S-prefixes
      s_prefix <- substr(cmd_str, 1, 2)
      match_idx <- match(s_prefix, ebops2010_s_map$s_prefix)
      if (!is.na(match_idx)) return(ebops2010_s_map$bpm6_category[match_idx])
      return(NA_character_)
    }

    # Handle C-prefix codes (C.1, C.1.1, etc.) — EBOPS 2002 style in EB format
    if (grepl("^C\\.", cmd_str)) return(NA_character_)

    # Handle numeric codes — check telecom subcodes first
    if (grepl("^9$", cmd_str)) return(NA_character_)     # Code "9" aggregate → skip
    if (grepl("^9\\.1", cmd_str)) return("Telecom_9_1")  # 9.1, 9.1.1, etc.
    if (grepl("^9\\.2", cmd_str)) return("Telecom_9_2")  # 9.2, 9.2.1, etc.
    if (grepl("^9\\.3", cmd_str)) return("Telecom_9_3")  # 9.3, 9.3.1, etc.

    # All other numeric codes: extract top-level integer
    top <- sub("^(\\d+).*$", "\\1", cmd_str)
    if (grepl("^\\d+$", top)) {
      match_idx <- match(top, ebops2010_map$top_code)
      if (!is.na(match_idx)) return(ebops2010_map$bpm6_category[match_idx])
    }

    return(NA_character_)
  }

  # --- EBOPS 2002 (EB02) ---
  if (grepl("EB02", classification)) {
    num_str <- sub("^(\\d+).*$", "\\1", cmd_str)
    if (!grepl("^\\d+$", num_str)) return(NA_character_)
    num_val <- as.integer(num_str)

    for (j in seq_len(nrow(ebops2002_ranges))) {
      if (num_val >= ebops2002_ranges$range_start[j] &&
          num_val <= ebops2002_ranges$range_end[j]) {
        return(ebops2002_ranges$bpm6_category[j])
      }
    }
    return(NA_character_)
  }

  return(NA_character_)
}

cat("EBOPS → BPM6 mapping functions ready.\n")
cat(sprintf("  EBOPS 2010 numeric mappings: %d (excl. code 9 → disaggregated)\n",
            nrow(ebops2010_map)))
cat(sprintf("  EBOPS 2010 S-prefix mappings: %d (excl. SI → disaggregated)\n",
            nrow(ebops2010_s_map)))
cat(sprintf("  EBOPS 2002 range mappings: %d\n", nrow(ebops2002_ranges)))
cat("  Telecom codes mapped to: Telecom_9_1, Telecom_9_2, Telecom_9_3\n")


--- Building EBOPS → BPM6 mapping (disaggregated telecom) ---
EBOPS → BPM6 mapping functions ready.
  EBOPS 2010 numeric mappings: 11 (excl. code 9 → disaggregated)
  EBOPS 2010 S-prefix mappings: 11 (excl. SI → disaggregated)
  EBOPS 2002 range mappings: 13
  Telecom codes mapped to: Telecom_9_1, Telecom_9_2, Telecom_9_3


<div style="font-size: 20px; line-height: 1.25;">
SECTION 4E — EXTRACT RAW TARIFF VALUES (Tariff_Raw)
</div>

<!-- ▶ MODIFICATION 2 — new section, no replacement. -->

The `Tariffs and trade defence measures applied on ICT goods` pillar contains
two numeric subpillars (effective tariff rate, zero-tariff coverage rate) where
the actual numeric value is stored in the `Description` column. Those values
are reported inconsistently — some look like decimals (`0.85`), others like
percentages without a sign (`8.5`), with mixed underlying types in the Excel
file (float, int, occasional string).

This section attaches a `Tariff_Raw` column to every row of `dti_active` that
preserves the original `Description` exactly as it appears, with no normalisation.
Non-tariff rows get `NA`. The format-normalisation decision is deferred — the
notebook stops at the report below and waits for a follow-up instruction.

The raw value is propagated down to the country-disaggregated rows produced in
Section 4B by joining on `dti_excel_row`, which is preserved through the
disaggregation step.

In [9]:
cat("\n--- Extracting raw tariff values (Modification 2) ---\n")

# Step 1: Build a tariff_raw lookup keyed by the original Excel row number,
# so the value follows the regulation through any later filtering/duplication.
tariff_pillar <- "Tariffs and trade defence measures applied on ICT goods"

tariff_lookup <- dti %>%
  filter(Pillar == tariff_pillar) %>%
  transmute(
    dti_excel_row = dti_excel_row,
    Subpillar     = Subpillar,
    Tariff_Raw    = as.character(Description)
  )

cat(sprintf("Tariff regulations found: %d\n", nrow(tariff_lookup)))
cat("Subpillar breakdown:\n")
print(table(tariff_lookup$Subpillar, useNA = "ifany"))

# Step 2: Attach Tariff_Raw to dti_active by joining on dti_excel_row.
# Drop only the lookup-side Subpillar column; we keep dti_active's own Subpillar.
dti_active <- dti_active %>%
  left_join(tariff_lookup %>% select(dti_excel_row, Tariff_Raw),
            by = "dti_excel_row")

# Step 3: Build a frequency report of raw value formats (numeric tariff
# subpillars only — the two subpillars where format normalisation matters).
numeric_tariff_subs <- c(
  "Effective tariff rate on ICT goods (applied weighted average)",
  "Coverage rate of zero-tariffs on ICT goods (%)"
)

tariff_numeric <- tariff_lookup %>% filter(Subpillar %in% numeric_tariff_subs)

# Categorise each raw value
classify_tariff_value <- function(v) {
  if (is.na(v) || v == "" || tolower(v) == "nan") return("NA / empty")
  v_trim <- trimws(v)

  # Pure numeric (integer or decimal)
  if (grepl("^-?[0-9]+(\\.[0-9]+)?$", v_trim)) {
    num <- suppressWarnings(as.numeric(v_trim))
    if (is.na(num)) return("unparseable_numeric")
    if (num >= 0 & num <= 1)   return("numeric_decimal_0_to_1")
    if (num >  1 & num <  100) return("numeric_number_1_to_100")
    if (num >= 100)            return("numeric_number_100_plus")
    return("numeric_negative")
  }

  # Numeric followed by percent sign
  if (grepl("^-?[0-9]+(\\.[0-9]+)?\\s*%$", v_trim)) return("string_with_percent")

  # Anything else is free-text
  "string_text"
}

tariff_numeric$fmt <- vapply(tariff_numeric$Tariff_Raw, classify_tariff_value,
                              character(1))

cat("\n=================================================================\n")
cat("  TARIFF_RAW FORMAT REPORT — numeric tariff subpillars only\n")
cat("=================================================================\n")
cat(sprintf("Total rows: %d\n", nrow(tariff_numeric)))

cat("\nFormat frequency:\n")
print(table(tariff_numeric$fmt, useNA = "ifany"))

cat("\n--- By subpillar ---\n")
for (sub in numeric_tariff_subs) {
  sub_df <- tariff_numeric %>% filter(Subpillar == sub)
  cat(sprintf("\n%s (%d rows):\n", sub, nrow(sub_df)))
  print(table(sub_df$fmt, useNA = "ifany"))
  cat("\nSample values per format:\n")
  for (f in unique(sub_df$fmt)) {
    samples <- head(sub_df$Tariff_Raw[sub_df$fmt == f], 6)
    cat(sprintf("  [%s] %s\n", f, paste(shQuote(samples), collapse = ", ")))
  }
}

cat(sprintf("\nNote: dti_active now has Tariff_Raw on every row (%d rows total). ",
            nrow(dti_active)))
cat(sprintf("Of those, %d carry a non-NA tariff value.\n",
            sum(!is.na(dti_active$Tariff_Raw))))


--- Extracting raw tariff values (Modification 2) ---
Tariff regulations found: 471
Subpillar breakdown:

                    Antidumping, countervailing duties, and safeguard measures on ICT goods 
                                                                                         37 
                                             Coverage rate of zero-tariffs on ICT goods (%) 
                                                                                        109 
                              Effective tariff rate on ICT goods (applied weighted average) 
                                                                                        164 
Participation in the WTO Information Technology Agreement (ITA) and 2015 expansion (ITA II) 
                                                                                        161 

  TARIFF_RAW FORMAT REPORT — numeric tariff subpillars only
Total rows: 273

Format frequency:

             NA / empty  numeric_decimal_0_to_1 nume

<div style="font-size: 25px; line-height: 1.25;">
SECTION 5 — LOAD COMTRADE DATA, MAP EBOPS, AND ADD FLAG COLUMNS
</div>

In [10]:
# --- 5.1 Load Comtrade data ---------------------------------------------------

cat("\n--- Loading Comtrade data ---\n")

if (exists("full_df") && is.data.frame(full_df) && nrow(full_df) > 0) {
  cat("Using in-memory full_df.\n")
  comtrade <- full_df
} else {
  cat(sprintf("Loading from CSV: %s\n", COMTRADE_CSV))
  comtrade <- read.csv(COMTRADE_CSV, stringsAsFactors = FALSE)
}
cat(sprintf("Comtrade dimensions: %d rows × %d columns\n", nrow(comtrade), ncol(comtrade)))

# --- 5.2 Normalise Comtrade reporter names to uppercase -----------------------

comtrade$reporter_upper <- toupper(comtrade$reporterDesc)

# --- 5.3 Check country matching -----------------------------------------------

dti_countries  <- sort(unique(dti_active$country_norm))
ct_countries   <- sort(unique(comtrade$reporter_upper))

matched     <- intersect(dti_countries, ct_countries)
dti_only    <- setdiff(dti_countries, ct_countries)
ct_only     <- setdiff(ct_countries, dti_countries)

cat(sprintf("\nCountry matching results:\n"))
cat(sprintf("  Matched:               %d\n", length(matched)))
cat(sprintf("  In DTI only:           %d\n", length(dti_only)))
cat(sprintf("  In Comtrade only:      %d\n", length(ct_only)))

if (length(dti_only) > 0) {
  cat("\n  Countries in DTI but NOT in Comtrade (no trade data to flag):\n")
  cat(paste("   ", dti_only, collapse = "\n"), "\n")
}
if (length(ct_only) > 0 && length(ct_only) <= 50) {
  cat("\n  Countries in Comtrade but NOT in DTI (will have all-zero flags):\n")
  cat(paste("   ", ct_only, collapse = "\n"), "\n")
} else if (length(ct_only) > 50) {
  cat(sprintf("\n  %d countries in Comtrade but NOT in DTI (too many to list).\n",
              length(ct_only)))
}

# --- 5.4 Map each Comtrade row's cmdCode to a BPM6 category ------------------

cat("\n--- Mapping EBOPS codes to BPM6 categories ---\n")

class_col <- if ("classificationCode" %in% names(comtrade)) {
  "classificationCode"
} else if ("classificationSearchCode" %in% names(comtrade)) {
  "classificationSearchCode"
} else {
  stop("Cannot find classification column in Comtrade data.")
}

comtrade$bpm6_category <- mapply(map_ebops_to_bpm6,
                                  comtrade$cmdCode,
                                  comtrade[[class_col]],
                                  USE.NAMES = FALSE)

mapped_count   <- sum(!is.na(comtrade$bpm6_category))
unmapped_count <- sum(is.na(comtrade$bpm6_category))

cat(sprintf("  Mapped to BPM6:       %s rows (%0.1f%%)\n",
            format(mapped_count, big.mark = ","),
            100 * mapped_count / nrow(comtrade)))
cat(sprintf("  Unmapped (agg/other): %s rows\n",
            format(unmapped_count, big.mark = ",")))
cat("\nBPM6 category distribution:\n")
print(table(comtrade$bpm6_category, useNA = "ifany"))


--- Loading Comtrade data ---
Loading from CSV: ../data/raw/data_result.csv
Comtrade dimensions: 1486955 rows × 47 columns

Country matching results:
  Matched:               47
  In DTI only:           117
  In Comtrade only:      9

  Countries in DTI but NOT in Comtrade (no trade data to flag):
    ALGERIA
    ANGOLA
    ARGENTINA
    ARMENIA
    BAHAMAS
    BAHRAIN
    BANGLADESH
    BARBADOS
    BELIZE
    BENIN
    BHUTAN
    BOLIVIA (PLURINATIONAL STATE OF)
    BOTSWANA
    BRAZIL
    BRUNEI DARUSSALAM
    BURKINA FASO
    BURUNDI
    CABO VERDE
    CAMBODIA
    CAMEROON
    CENTRAL AFRICAN REPUBLIC
    CHAD
    CHINA
    COLOMBIA
    COMOROS
    CONGO
    COSTA RICA
    CÔTE D'IVOIRE
    CUBA
    DEM. REP. OF THE CONGO
    DJIBOUTI
    DOMINICAN REPUBLIC
    ECUADOR
    EGYPT
    EL SALVADOR
    EQUATORIAL GUINEA
    ERITREA
    ESWATINI
    ETHIOPIA
    EUROPEAN UNION
    FIJI
    GABON
    GAMBIA
    GHANA
    GUATEMALA
    GUINEA
    GUINEA-BISSAU
    GUYANA
    HAITI
    H

In [11]:
# Quick overview of the matched countries (DTI ∩ Comtrade) — laid out in a grid
cat(sprintf("Matched countries (%d):\n\n", length(matched)))
print(matrix(c(matched, rep("", (5 - length(matched) %% 5) %% 5)),
             ncol = 5, byrow = TRUE,
             dimnames = list(NULL, paste0("col", 1:5))), quote = FALSE)

Matched countries (47):

      col1       col2                col3                 col4       
 [1,] AUSTRALIA  AUSTRIA             AZERBAIJAN           BELGIUM    
 [2,] CANADA     CHILE               CHINA, HONG KONG SAR CROATIA    
 [3,] CZECHIA    DENMARK             ESTONIA              FINLAND    
 [4,] GEORGIA    GERMANY             GREECE               HUNGARY    
 [5,] ITALY      JAPAN               KYRGYZSTAN           LATVIA     
 [6,] LUXEMBOURG MALTA               NETHERLANDS          NEW ZEALAND
 [7,] POLAND     PORTUGAL            REP. OF KOREA        ROMANIA    
 [8,] SINGAPORE  SLOVAKIA            SLOVENIA             SPAIN      
 [9,] THAILAND   TRINIDAD AND TOBAGO TUNISIA              TÜRKIYE    
[10,] USA        UZBEKISTAN                                          
      col5              
 [1,] BULGARIA          
 [2,] CYPRUS            
 [3,] FRANCE            
 [4,] IRELAND           
 [5,] LITHUANIA         
 [6,] NORWAY            
 [7,] RUSSIAN FEDERATION
 [8,]

<div style="font-size: 20px; line-height: 1.25;">
SECTION 5 (continued) — PRE-COMPUTE LOOKUPS AND JOIN
</div>

In [12]:
# --- 5.5 Pre-compute regulation lookup tables (one per direction) -------------

cat("\n--- Pre-computing regulation lookup tables ---\n")

ct_years <- sort(unique(as.integer(comtrade$refYear)))
cat(sprintf("Comtrade year range: %d–%d\n", min(ct_years), max(ct_years)))

# Expand each regulation across its active years
expand_regulation <- function(reg_row) {
  start_yr <- reg_row$activation_year
  end_yr   <- ifelse(is.na(reg_row$expiry_year), max(ct_years), reg_row$expiry_year)
  start_yr <- max(start_yr, min(ct_years))
  end_yr   <- min(end_yr, max(ct_years))
  if (start_yr > end_yr) return(NULL)

  years <- start_yr:end_yr

  affected <- character(0)
  for (cat_name in bpm6_categories) {
    effect_val <- reg_row[[paste0(cat_name, "_effect")]]
    if (!is.na(effect_val) && effect_val == 1) {
      affected <- c(affected, cat_name)
    }
  }
  if (length(affected) == 0) return(NULL)

  expand.grid(
    country_norm    = reg_row$country_norm,
    year            = years,
    bpm6_category   = affected,
    col_assignment  = reg_row$col_assignment,
    direction_class = reg_row$direction_class,
    stringsAsFactors = FALSE
  )
}

cat("Expanding regulations across years and categories...\n")
flush.console()

expanded_list <- vector("list", nrow(dti_active))
for (i in seq_len(nrow(dti_active))) {
  expanded_list[[i]] <- expand_regulation(dti_active[i, ])
}

expanded_all <- bind_rows(compact(expanded_list))
cat(sprintf("Expanded regulation-year-category records: %s\n",
            format(nrow(expanded_all), big.mark = ",")))
cat(sprintf("  Negative: %s\n",
            format(sum(expanded_all$direction_class == "negative"), big.mark = ",")))
cat(sprintf("  Positive: %s\n",
            format(sum(expanded_all$direction_class == "positive"), big.mark = ",")))

# --- 5.6 Build lookup tables for negative and positive separately -------------

build_lookup_for_direction <- function(direction_label, suffix) {
  cat(sprintf("\n  Building %s lookup...\n", direction_label))
  flush.console()

  dir_data <- expanded_all %>% filter(direction_class == direction_label)

  if (nrow(dir_data) == 0) {
    cat(sprintf("    No %s regulations found.\n", direction_label))
    return(NULL)
  }

  lookup_list <- list()
  for (col_id in all_flag_cols) {
    subset_df <- dir_data %>%
      filter(col_assignment == col_id) %>%
      group_by(country_norm, year, bpm6_category) %>%
      summarise(effect = 1L, count = n(), .groups = "drop")

    if (nrow(subset_df) > 0) {
      names(subset_df)[names(subset_df) == "effect"] <- paste0(col_id, "_", suffix, "_effect")
      names(subset_df)[names(subset_df) == "count"]  <- paste0(col_id, "_", suffix, "_count")
      lookup_list[[col_id]] <- subset_df
    }
  }

  lookup <- NULL
  for (col_id in all_flag_cols) {
    if (!is.null(lookup_list[[col_id]])) {
      if (is.null(lookup)) {
        lookup <- lookup_list[[col_id]]
      } else {
        lookup <- full_join(lookup, lookup_list[[col_id]],
                            by = c("country_norm", "year", "bpm6_category"))
      }
    }
  }

  if (is.null(lookup)) return(NULL)

  expected_effect <- paste0(all_flag_cols, "_", suffix, "_effect")
  expected_count  <- paste0(all_flag_cols, "_", suffix, "_count")
  for (col in c(expected_effect, expected_count)) {
    if (col %in% names(lookup)) {
      lookup[[col]][is.na(lookup[[col]])] <- 0L
    } else {
      lookup[[col]] <- 0L
    }
  }

  cat(sprintf("    %s lookup: %s rows\n", direction_label,
              format(nrow(lookup), big.mark = ",")))
  return(lookup)
}

lookup_neg <- build_lookup_for_direction("negative", "neg")
lookup_pos <- build_lookup_for_direction("positive", "pos")

# --- 5.7 Join lookup tables to Comtrade data ----------------------------------

cat("\n--- Joining flags to Comtrade data ---\n")

comtrade$year <- as.integer(comtrade$refYear)

neg_effect_cols <- paste0(all_flag_cols, "_neg_effect")
neg_count_cols  <- paste0(all_flag_cols, "_neg_count")
pos_effect_cols <- paste0(all_flag_cols, "_pos_effect")
pos_count_cols  <- paste0(all_flag_cols, "_pos_count")

all_new_cols <- c(neg_effect_cols, neg_count_cols, pos_effect_cols, pos_count_cols)

cat("  Joining negative effects...\n")
flush.console()

if (!is.null(lookup_neg)) {
  comtrade_merged <- comtrade %>%
    left_join(lookup_neg,
              by = c("reporter_upper" = "country_norm",
                     "year"           = "year",
                     "bpm6_category"  = "bpm6_category"))
} else {
  comtrade_merged <- comtrade
}

cat("  Joining positive effects...\n")
flush.console()

if (!is.null(lookup_pos)) {
  comtrade_merged <- comtrade_merged %>%
    left_join(lookup_pos,
              by = c("reporter_upper" = "country_norm",
                     "year"           = "year",
                     "bpm6_category"  = "bpm6_category"))
}

for (col in all_new_cols) {
  if (col %in% names(comtrade_merged)) {
    comtrade_merged[[col]][is.na(comtrade_merged[[col]])] <- 0L
  } else {
    comtrade_merged[[col]] <- 0L
  }
}

cat(sprintf("\nMerged dataset: %s rows × %d columns\n",
            format(nrow(comtrade_merged), big.mark = ","),
            ncol(comtrade_merged)))
cat(sprintf("New columns added: %d (%d neg effect + %d neg count + %d pos effect + %d pos count)\n",
            length(all_new_cols),
            length(neg_effect_cols), length(neg_count_cols),
            length(pos_effect_cols), length(pos_count_cols)))

cat("\nColumn layout:\n")
cat(sprintf("  Trade data columns:      1 – %d\n", ncol(comtrade) + 1))
cat(sprintf("  Negative effect block:   %d columns (%s ... %s)\n",
            length(neg_effect_cols) + length(neg_count_cols),
            neg_effect_cols[1], neg_count_cols[length(neg_count_cols)]))
cat(sprintf("  Positive effect block:   %d columns (%s ... %s)\n",
            length(pos_effect_cols) + length(pos_count_cols),
            pos_effect_cols[1], pos_count_cols[length(pos_count_cols)]))


--- Pre-computing regulation lookup tables ---
Comtrade year range: 2000–2024
Expanding regulations across years and categories...
Expanded regulation-year-category records: 128,294
  Negative: 91,228
  Positive: 37,066

  Building negative lookup...
    negative lookup: 27,908 rows

  Building positive lookup...
    positive lookup: 17,153 rows

--- Joining flags to Comtrade data ---
  Joining negative effects...
  Joining positive effects...

Merged dataset: 1,486,955 rows × 114 columns
New columns added: 64 (16 neg effect + 16 neg count + 16 pos effect + 16 pos count)

Column layout:
  Trade data columns:      1 – 51
  Negative effect block:   32 columns (p6_1_neg_effect ... p12_neg_count)
  Positive effect block:   32 columns (p6_1_pos_effect ... p12_pos_count)


<div style="font-size: 25px; line-height: 1.25;">
SECTION 6 — VALIDATION
</div>

In [13]:
cat("\n╔══════════════════════════════════════════════════════════════╗\n")
cat("║   VALIDATION CHECKS                                        ║\n")
cat("╚══════════════════════════════════════════════════════════════╝\n")

cat("\n--- Negative effect flag summaries ---\n")
for (col_id in all_flag_cols) {
  ecol <- paste0(col_id, "_neg_effect")
  ones  <- sum(comtrade_merged[[ecol]] == 1, na.rm = TRUE)
  zeros <- sum(comtrade_merged[[ecol]] == 0, na.rm = TRUE)
  cat(sprintf("  %-20s: 1s = %s, 0s = %s\n",
              ecol, format(ones, big.mark = ","), format(zeros, big.mark = ",")))
}

cat("\n--- Positive effect flag summaries ---\n")
for (col_id in all_flag_cols) {
  ecol <- paste0(col_id, "_pos_effect")
  ones  <- sum(comtrade_merged[[ecol]] == 1, na.rm = TRUE)
  zeros <- sum(comtrade_merged[[ecol]] == 0, na.rm = TRUE)
  cat(sprintf("  %-20s: 1s = %s, 0s = %s\n",
              ecol, format(ones, big.mark = ","), format(zeros, big.mark = ",")))
}

cat("\n--- Negative count column summaries ---\n")
for (col_id in all_flag_cols) {
  ccol <- paste0(col_id, "_neg_count")
  vals <- comtrade_merged[[ccol]]
  cat(sprintf("  %-20s: min=%d, max=%d, mean=%.2f, non-zero=%s\n",
              ccol, min(vals), max(vals), mean(vals),
              format(sum(vals > 0), big.mark = ",")))
}

cat("\n--- Positive count column summaries ---\n")
for (col_id in all_flag_cols) {
  ccol <- paste0(col_id, "_pos_count")
  vals <- comtrade_merged[[ccol]]
  cat(sprintf("  %-20s: min=%d, max=%d, mean=%.2f, non-zero=%s\n",
              ccol, min(vals), max(vals), mean(vals),
              format(sum(vals > 0), big.mark = ",")))
}

cat("\n--- Consistency: count > 0 where effect == 0? ---\n")
issues <- 0
for (suffix in c("neg", "pos")) {
  for (col_id in all_flag_cols) {
    ecol <- paste0(col_id, "_", suffix, "_effect")
    ccol <- paste0(col_id, "_", suffix, "_count")
    bad <- sum(comtrade_merged[[ccol]] > 0 & comtrade_merged[[ecol]] == 0, na.rm = TRUE)
    if (bad > 0) {
      cat(sprintf("  ⚠ %s_%s: %d rows with count > 0 but effect == 0\n",
                  col_id, suffix, bad))
      issues <- issues + bad
    }
  }
}
if (issues == 0) cat("  ✓ All consistent across both negative and positive blocks.\n")

cat("\n--- Expiry date check ---\n")
regs_with_expiry <- dti_active %>% filter(!is.na(expiry_year))
cat(sprintf("  Regulations with expiry dates: %d\n", nrow(regs_with_expiry)))
cat("  ✓ Expiry dates are enforced in the expansion step.\n")

cat("\n--- Unmatched reporters ---\n")
cat(sprintf("  DTI countries with no Comtrade match: %d\n", length(dti_only)))
if (length(dti_only) > 0) cat(paste("   ", dti_only, collapse = "\n"), "\n")

cat("\n--- Unmapped EBOPS codes ---\n")
unmapped_codes <- comtrade_merged %>%
  filter(is.na(bpm6_category)) %>%
  distinct(cmdCode) %>%
  pull(cmdCode)
cat(sprintf("  EBOPS codes with no BPM6 mapping: %d\n", length(unmapped_codes)))
if (length(unmapped_codes) > 0 && length(unmapped_codes) <= 30) {
  cat(paste("   ", sort(unmapped_codes), collapse = "\n"), "\n")
  cat("  (These are typically aggregate codes like TOTAL, S, 200, etc.)\n")
}


╔══════════════════════════════════════════════════════════════╗
║   VALIDATION CHECKS                                        ║
╚══════════════════════════════════════════════════════════════╝

--- Negative effect flag summaries ---
  p6_1_neg_effect     : 1s = 7,670, 0s = 1,479,285
  p6_2_neg_effect     : 1s = 22,484, 0s = 1,464,471
  p6_3_neg_effect     : 1s = 0, 0s = 1,486,955
  p6_4_neg_effect     : 1s = 29,268, 0s = 1,457,687
  p6_5_neg_effect     : 1s = 0, 0s = 1,486,955
  p1_neg_effect       : 1s = 17,486, 0s = 1,469,469
  p2_neg_effect       : 1s = 105,475, 0s = 1,381,480
  p3_neg_effect       : 1s = 110,449, 0s = 1,376,506
  p4_neg_effect       : 1s = 190,090, 0s = 1,296,865
  p5_neg_effect       : 1s = 40,917, 0s = 1,446,038
  p7_neg_effect       : 1s = 78,000, 0s = 1,408,955
  p8_neg_effect       : 1s = 48,996, 0s = 1,437,959
  p9_neg_effect       : 1s = 69,429, 0s = 1,417,526
  p10_neg_effect      : 1s = 228,443, 0s = 1,258,512
  p11_neg_effect      : 1s = 48,277, 0s = 1,4

<div style="font-size: 25px; line-height: 1.25;">
SECTION 7 — EXPORT
</div>

In [ ]:
cat("\n--- Exporting merged dataset ---\n")

cols_to_drop <- c("reporter_upper", "year")
comtrade_export <- comtrade_merged %>%
  select(-any_of(cols_to_drop))

write.csv(comtrade_export, file = OUTPUT_MERGED, row.names = FALSE,
          fileEncoding = "UTF-8")

file_size_mb <- round(file.info(OUTPUT_MERGED)$size / 1e6, 1)

cat("\n╔══════════════════════════════════════════════════════════════╗\n")
cat("║   EXPORT COMPLETE                                          ║\n")
cat("╠══════════════════════════════════════════════════════════════╣\n")
cat(sprintf("║  File path : %s\n", OUTPUT_MERGED))
cat(sprintf("║  Rows      : %s\n", format(nrow(comtrade_export), big.mark = ",")))
cat(sprintf("║  Columns   : %d\n", ncol(comtrade_export)))
cat(sprintf("║  File size : %.1f MB\n", file_size_mb))
cat("╠══════════════════════════════════════════════════════════════╣\n")
cat("║  Non-zero entries per NEGATIVE flag column:                ║\n")
for (col_id in all_flag_cols) {
  ecol <- paste0(col_id, "_neg_effect")
  n <- sum(comtrade_export[[ecol]] == 1, na.rm = TRUE)
  cat(sprintf("║    %-22s: %s\n", ecol, format(n, big.mark = ",")))
}
cat("║  Non-zero entries per POSITIVE flag column:                ║\n")
for (col_id in all_flag_cols) {
  ecol <- paste0(col_id, "_pos_effect")
  n <- sum(comtrade_export[[ecol]] == 1, na.rm = TRUE)
  cat(sprintf("║    %-22s: %s\n", ecol, format(n, big.mark = ",")))
}
cat("╚══════════════════════════════════════════════════════════════╝\n")

full_df_merged <- comtrade_export
cat("\nDone. Merged dataset available as 'full_df_merged'. ✓\n")


--- Exporting merged dataset ---

╔══════════════════════════════════════════════════════════════╗
║   EXPORT COMPLETE                                          ║
╠══════════════════════════════════════════════════════════════╣
║  File path : ../data/final_dataset/comtrade_dti_final.csv
║  Rows      : 1,486,955
║  Columns   : 112
║  File size : 635.0 MB
╠══════════════════════════════════════════════════════════════╣
║  Non-zero entries per NEGATIVE flag column:                ║
║    p6_1_neg_effect       : 7,670
║    p6_2_neg_effect       : 22,484
║    p6_3_neg_effect       : 0
║    p6_4_neg_effect       : 29,268
║    p6_5_neg_effect       : 0
║    p1_neg_effect         : 17,486
║    p2_neg_effect         : 105,475
║    p3_neg_effect         : 110,449
║    p4_neg_effect         : 190,090
║    p5_neg_effect         : 40,917
║    p7_neg_effect         : 78,000
║    p8_neg_effect         : 48,996
║    p9_neg_effect         : 69,429
║    p10_neg_effect        : 228,443
║    p11_neg_effect

: 

In [ ]:
options(width = 300, dplyr.width = Inf)
print(as.data.frame(tail(full_df_merged, 10)))